# Cycle Design Point
### Roadrunner propulsion reverse-engineering study · notebook 9 of 15

**Purpose.** Run pyCycle at the engine sizing condition, OP1 (ADR-0009): sea-level static on the
mission's hot day, with the thrust each engine must give for vertical launch. It answers:

> *What single-spool turbojet, in air flow, specific thrust, fuel flow and station
> temperatures, delivers the launch thrust, and how much does that depend on what we had to
> assume?*

The engine is *required*, not designed and not yet a catalogue engine (ADR-0023, ADR-0030).
Thrust comes from the launch constraint of the converged design point (`solve_design_point()`),
turbine inlet temperature from the catalogue-anchored technology level (`out/validation.yaml`),
pressure ratio and component losses from `config/cycle.yaml`. At fixed $T_{t4}$ and pressure
ratio, thrust is proportional to air flow, so air flow follows from the thrust; a second run
verifies the closure. Station numbering is ARP755.

---
## 0 · Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import yaml

from roadrunner_re import cycle_outputs as co
from roadrunner_re.analysis_inputs import load_analysis_inputs
from roadrunner_re.cycle.design import DesignInputs, one_at_a_time, size_design_point
from roadrunner_re.cycle.turbojet import FlightCondition, load_cycle_settings
from roadrunner_re.design_point import solve_design_point
from roadrunner_re.notebook import nb_setup
from roadrunner_re.plots.cycle import station_figure, tornado_figure
from roadrunner_re.uncertain import Uncertain

paths = nb_setup()
analysis = load_analysis_inputs()
design = solve_design_point(analysis)
settings = load_cycle_settings()
validation = yaml.safe_load((paths.out / "validation.yaml").read_text(encoding="utf-8"))
technology = yaml.safe_load((paths.out / "technology.yaml").read_text(encoding="utf-8"))
t4 = validation["anchored_technology"]["Tt4_K"]
Tt4 = Uncertain(t4["median"], t4["low"], t4["high"])

---
## 1 · The requirement

Vertical launch needs thrust equal to the launch margin times weight. The hot day lowers the
algebraic lapse, so the sea-level static thrust the engines must give *on that day* is the
launch-constraint thrust loading times the hot-day lapse times weight, shared between the two
engines. (First iteration: the lapse is the algebraic one of ADR-0029; the next notebook replaces
it with the cycle's own.)

In [ ]:
condition = FlightCondition(0.0, 0.0, analysis.mission.dT_isa_K.value)
nominal = DesignInputs(design.launch_thrust_per_engine_N, settings.pressure_ratio_when_unpublished.value,
                       Tt4.value, condition, settings.nominal_losses())
print(f"take-off mass {design.takeoff_mass_kg:.0f} kg, {analysis.engine_count} engines, hot day ISA {condition.dT_isa_K:+.0f} K")
print(f"thrust required per engine {nominal.thrust_N:.1f} N (sea-level static, hot day)")
print(f"Tt4 {Tt4.value:.0f} K ({Tt4.low:.0f}-{Tt4.high:.0f}), pressure ratio {nominal.pressure_ratio} "
      f"({settings.pressure_ratio_when_unpublished.low}-{settings.pressure_ratio_when_unpublished.high})")

---
## 2 · The design point

In [ ]:
result = size_design_point(settings, nominal)
print(f"air flow          {result.air_mass_flow_kg_s:8.4f} kg/s")
print(f"specific thrust   {result.specific_thrust_N_s_kg:8.1f} N s/kg")
print(f"ideal fuel flow   {result.fuel_flow_kg_s * 1000:8.2f} g/s   SFC {result.sfc_kg_N_s:.3e} kg/(N s)")
print(f"fuel-air ratio    {result.fuel_air_ratio:8.4f}")
print(f"exit velocity     {result.exit_velocity_m_s:8.1f} m/s,  nozzle pressure ratio {result.nozzle_pressure_ratio:.2f}")
print()
print(f"{'station':>8s}{'Tt [K]':>10s}{'Pt [kPa]':>11s}{'W [kg/s]':>10s}")
for name, (T, P, W) in result.stations.items():
    print(f"{name:>8s}{T:10.1f}{P / 1e3:11.2f}{W:10.4f}")

In [ ]:
fig = station_figure(result, paths.figures / "cycle_stations.png")

Station 2 is the compressor face, 3 the compressor exit, 4 the turbine inlet, 5 the turbine
exit and 8 the nozzle throat (convergent nozzle). The turbine pressure ratio is whatever balances
compressor power; the compressor pressure ratio is the assumed input.

---
## 3 · Against the catalogue technology

The model's specific thrust and SFC against the range of the catalogue engines
(`out/technology.yaml`). The SFC here is the *ideal* value; real engines burn more.

In [ ]:
bounds = technology["bounds"]
for key, value in (("specific_thrust_N_s_kg", result.specific_thrust_N_s_kg), ("sfc_kg_N_s", result.sfc_kg_N_s)):
    b = bounds[key]
    print(f"{key:26s} model {value:10.4g}   catalogue {b['low']:.4g} to {b['high']:.4g}   inside: {b['low'] <= value <= b['high']}")

---
## 4 · What the assumptions do

Each uncertain input moves to its bounds with the others at nominal (ADR-0015). The bars show
the spread in the engine's size (air flow) and ideal SFC.

In [ ]:
oat = one_at_a_time(settings, nominal, Tt4, settings.pressure_ratio_when_unpublished)
fig = tornado_figure(oat, result, paths.figures / "cycle_assumptions.png")
for name in dict.fromkeys(r["input"] for r in oat):
    rows = [r for r in oat if r["input"] == name]
    lo, hi = (min(r["air_mass_flow_kg_s"] for r in rows), max(r["air_mass_flow_kg_s"] for r in rows))
    print(f"{name:28s} air flow {lo:.4f} to {hi:.4f} kg/s")

---
## 5 · Handoff

In [ ]:
content = co.cycle_handoff(nominal, result, engine_count=analysis.engine_count, Tt4=Tt4,
                           pressure_ratio=settings.pressure_ratio_when_unpublished, oat=oat, technology_bounds=bounds)
handoff = co.write_cycle_handoff(content, paths.out / "cycle.yaml")
print(f"wrote {handoff.relative_to(paths.repo)}")

---
## Findings

- **The required engine is a small one**: about one kilogram per second of air per engine, with
  a specific thrust inside the catalogue range, so the technology assumption is self-consistent.
- **Turbine inlet temperature dominates the spread**: it moves specific thrust and so the air flow
  the thrust needs more than any loss; compressor and turbine efficiency come next.
- **Fuel flow is the ideal value.** A real engine burns more; the first-iteration mission used the
  catalogue SFC, which already includes that.
- **Standing findings:** the Tt4 anchor depends on the assumed efficiencies (notebook 5); no
  installation loss is applied; the thrust requirement is the first-iteration launch thrust and
  moves with the constraint analysis.